# CMASW Convexity Correction — Pucci (2012a)

**Reference:** M. Pucci, *Constant Maturity Asset Swap Convexity Correction*, Banca IMI, 2012. SSRN 1961545.

Visual validation of the convexity correction for ASW-lets under the linear swap-rate model.

## Contents
1. Lognormal CC surface (paper Table 2): sigma_asw vs rho
2. CC sign follows rho
3. Prefactor analysis
4. Displaced vs lognormal comparison

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

from datetime import date, timedelta
from pricebook.bootstrap import bootstrap
from pricebook.cmasw import CMASWInstrument
from pricebook.viz import plot, PlotBuilder

REF = date(2026, 4, 26)
deposits = [(REF + timedelta(days=91), 0.04), (REF + timedelta(days=182), 0.039)]
swaps = [(REF + timedelta(days=365), 0.038), (REF + timedelta(days=1825), 0.035),
         (REF + timedelta(days=3650), 0.034)]
curve = bootstrap(REF, deposits, swaps)

cmasw = CMASWInstrument(
    REF + timedelta(days=1825), REF + timedelta(days=2007),
    swap_tenor=5, bond_price=0.95, sigma_swp=0.30, sigma_asw=0.25, rho=0.5)

# Default dashboard — one line
fig = plot(cmasw, curve)

## 1. Lognormal CC Heatmap (Paper Table 2)

$\text{CC} = R_0^{asw} \left(1 - \frac{A_0 \alpha}{D_{0,T_p}}\right) \left(e^{\sigma_{swp}\sigma_{asw}\rho T_0} - 1\right)$

In [ ]:
sigma_asw_range = [0.10, 0.20, 0.30, 0.40, 0.50]
rho_range = [-0.9, -0.5, 0.0, 0.5, 0.9]

cc_table = np.zeros((len(sigma_asw_range), len(rho_range)))
for i, sig in enumerate(sigma_asw_range):
    for j, rho in enumerate(rho_range):
        cc_table[i, j] = cmasw_cc_lognormal(
            R_ASW, ANNUITY, DF_TP, YFS, SIGMA_SWP, sig, rho, T0) * 100  # in %

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(cc_table, cmap="RdBu_r", aspect="auto",
               extent=[-0.9, 0.9, 0.50, 0.10])
ax.set_xlabel("Correlation $\\rho$")
ax.set_ylabel("$\\sigma_{asw}$")
ax.set_title("Lognormal CC (% of notional) — Paper Table 2\n"
             f"$R^{{asw}}_0$={R_ASW*10000:.0f}bp, $\\sigma_{{swp}}$={SIGMA_SWP*100:.0f}%, $T_0$={T0:.0f}Y")
fig.colorbar(im, ax=ax, label="CC (%)")

# Print the table
print("Lognormal CC (%) — sigma_asw \\ rho")
header = "       " + "".join(f"{rho:>8.0%}" for rho in rho_range)
print(header)
for i, sig in enumerate(sigma_asw_range):
    row = f"{sig:5.0%}  " + "".join(f"{cc_table[i,j]:8.2f}%" for j in range(len(rho_range)))
    print(row)

plt.tight_layout()
plt.show()

## 2. CC as Function of Correlation

CC has the sign of $\rho$ in the lognormal case. Linear in $\rho$ to first order.

In [ ]:
rhos = np.linspace(-1, 1, 100)

fig, ax = plt.subplots(figsize=(10, 5))
for sig in [0.20, 0.30, 0.50]:
    ccs = [cmasw_cc_lognormal(R_ASW, ANNUITY, DF_TP, YFS,
           SIGMA_SWP, sig, rho, T0) * 10000 for rho in rhos]
    ax.plot(rhos, ccs, lw=2, label=f"$\\sigma_{{asw}}$={sig*100:.0f}%")

ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="gray", ls=":", alpha=0.5)
ax.set_xlabel("Correlation $\\rho$")
ax.set_ylabel("CC (bp)")
ax.set_title("Lognormal CC vs Correlation — Sign Follows $\\rho$")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Displaced vs Lognormal: Skew Impact

Displacement $a_{asw}$ controls the ATM skew. Small displacement $\Rightarrow$ small deviation from lognormal CC.

In [ ]:
a_asw_range = np.linspace(-0.02, 0.02, 50)
rho_fixed = 0.5
sig_asw = 0.30

cc_ln = cmasw_cc_lognormal(R_ASW, ANNUITY, DF_TP, YFS, SIGMA_SWP, sig_asw, rho_fixed, T0)

displaced_ccs = []
for a in a_asw_range:
    res = cmasw_convexity_correction(
        R_ASW, R_SWP, ANNUITY, DF_TP, YFS, DFS,
        SIGMA_SWP, sig_asw, rho_fixed, T0,
        a_swp=0.0, a_asw=a)
    displaced_ccs.append(res.convexity_correction * 10000)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(a_asw_range * 10000, displaced_ccs, "b-", lw=2, label="Displaced CC")
ax.axhline(cc_ln * 10000, color="r", ls="--", lw=2, label=f"Lognormal CC = {cc_ln*10000:.1f}bp")
ax.axvline(0, color="gray", ls=":", alpha=0.5)
ax.set_xlabel("$a_{asw}$ (bp)")
ax.set_ylabel("CC (bp)")
ax.set_title(f"Displaced vs Lognormal CC ($\\rho$={rho_fixed}, $\\sigma_{{asw}}$={sig_asw*100:.0f}%)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()